# Pikachu Robust RL - Colab T4
This notebook pins both repositories, verifies the production engine, resumes only verified episode-boundary checkpoints from Drive, and keeps validation separate from any sealed final set.

Private repository setup: add a GitHub token to Colab Secrets as `GITHUB_TOKEN` and enable notebook access. A fine-grained token with read-only Contents access is preferred; a classic PAT requires the `repo` scope. The token is passed through a temporary Git HTTP header and is never written to the clone URL or Git config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_URL = 'https://github.com/jimin326/skku_pikachu.git'
PROJECT_ROOT = '/content/skku_pikachu'
PROJECT_REF = 'robust-rl-colab'  # pinned branch containing the RL system
GAME_URL = 'https://github.com/SKKU-x-HYU-SW-Competition/leonyi-volleyball.git'
GAME_COMMIT = '1f3cecb90aca174ffc42ac6be4c384cc725d9e91'
TRAINING_SEED = 20260903
EXPERIMENT_ID = f'bc_ppo_seed{TRAINING_SEED}'
RUN_BC = True
TOTAL_STEPS = 2_000_000
BC_DATASET = '/content/drive/MyDrive/pikachu_rl/bc/v4_500k.jsonl'
BC_MODEL = '/content/drive/MyDrive/pikachu_rl/bc/v4_500k_ff.pt'
RECOVERY = f'/content/drive/MyDrive/pikachu_rl/{EXPERIMENT_ID}/checkpoints'
LOCAL_CHECKPOINT_DIR = f'/content/checkpoints/{EXPERIMENT_ID}'


In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import userdata
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a read-only GITHUB_TOKEN in Colab Secrets and enable notebook access') from exc
if not github_token:
    raise RuntimeError('Colab Secret GITHUB_TOKEN is empty')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env = os.environ.copy()
git_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'Authorization: Basic {basic_auth}')
project_path = Path(PROJECT_ROOT)
if not (project_path / '.git').is_dir():
    if project_path.exists() and any(project_path.iterdir()):
        raise RuntimeError(f'{project_path} exists but is not a Git checkout; restart the runtime or clear that directory')
    subprocess.run(['git','clone','--branch',PROJECT_REF,'--single-branch',PROJECT_URL,PROJECT_ROOT], check=True, env=git_env)
subprocess.run(['git','-C',PROJECT_ROOT,'fetch','origin',PROJECT_REF], check=True, env=git_env)
subprocess.run(['git','-C',PROJECT_ROOT,'checkout','-B',PROJECT_REF,'origin/'+PROJECT_REF], check=True, env=git_env)
github_token = basic_auth = git_env = None  # discard notebook references to credentials
game_root = Path('/content/leonyi-volleyball')
if not (game_root / '.git').is_dir():
    if game_root.exists() and any(game_root.iterdir()):
        raise RuntimeError(f'{game_root} exists but is not a Git checkout; restart the runtime or clear that directory')
    subprocess.run(['git','clone',GAME_URL,str(game_root)], check=True)
subprocess.run(['git','-C',str(game_root),'checkout',GAME_COMMIT], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-rl.txt'], check=True)
subprocess.run(['node','scripts/setup_rl_engine.mjs',str(game_root)], check=True)
print('Prepared', PROJECT_REF, 'at', subprocess.run(['git','rev-parse','HEAD'], cwd=PROJECT_ROOT, check=True, text=True, capture_output=True).stdout.strip())


In [ ]:
import subprocess, sys
checks = [
    ['node','bot-dev/rl/physics_clamp_smoke.mjs'],
    ['node','bot-dev/rl/production_differential.mjs','--game-root','/content/leonyi-volleyball'],
    ['node','bot-dev/rl/env_smoke.mjs'],
    [sys.executable,'bot-dev/rl/bridge_smoke.py'],
    [sys.executable,'bot-dev/rl/ppo_tests.py'],
    [sys.executable,'bot-dev/rl/eval/test_schema.py'],
    [sys.executable,'bot-dev/rl/eval/test_stats.py'],
    ['node','bot-dev/rl/eval/paired_eval_smoke.mjs'],
]
for command in checks:
    print('RUN', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime before training'
print(torch.cuda.get_device_name(0))


In [ ]:
# v4 behavior-cloning initialization. Existing complete artifacts are reused after reconnect.
import subprocess, sys
from pathlib import Path
bc_dataset = Path(BC_DATASET)
bc_metadata = Path(BC_DATASET + '.meta.json')
bc_model = Path(BC_MODEL)
bc_dataset.parent.mkdir(parents=True, exist_ok=True)
if RUN_BC:
    if not (bc_dataset.is_file() and bc_metadata.is_file()):
        subprocess.run(['node','bot-dev/rl/collect_bc.mjs','--decisions=500000',f'--output={bc_dataset}'], cwd=PROJECT_ROOT, check=True)
    if not bc_model.is_file():
        subprocess.run([sys.executable,'bot-dev/rl/bc_pretrain.py',str(bc_dataset),str(bc_model),'--epochs=10','--device=cuda'], cwd=PROJECT_ROOT, check=True)
    print('BC ready:', bc_model)


In [ ]:
import hashlib, json, os
from pathlib import Path
latest = Path(RECOVERY) / 'latest.json'
resume_args = []
if latest.is_file():
    pointer = json.loads(latest.read_text())
    checkpoint = Path(RECOVERY) / pointer['checkpoint']
    assert checkpoint.is_file(), f'Missing recovery checkpoint: {checkpoint}'
    assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == pointer['sha256']
    resume_args = ['--resume', str(checkpoint)]
    resume_mode = f"resume step {pointer['globalStep']}"
elif RUN_BC:
    assert Path(BC_MODEL).is_file(), f'Missing BC model: {BC_MODEL}'
    resume_args = ['--initial-model', BC_MODEL]
    resume_mode = 'new PPO run initialized from v4 BC'
else:
    resume_mode = 'new PPO run from random initialization'
print(resume_mode, resume_args)


In [ ]:
# Benchmark 8/16/32 total envs first; Node physics/IPC, not VRAM, is usually the bottleneck.
from pathlib import Path
import subprocess, sys
project_root = Path(PROJECT_ROOT)
train_script = project_root / 'bot-dev/rl/ppo_train.py'
assert train_script.is_file(), f'Missing {train_script}; rerun the clone/setup cell and verify PROJECT_REF={PROJECT_REF!r}'
checked_out = subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'], cwd=project_root, check=True, text=True, capture_output=True).stdout.strip()
assert checked_out == PROJECT_REF, f'Expected branch {PROJECT_REF!r}, found {checked_out!r}; rerun the clone/setup cell'
args = [sys.executable,str(train_script),'--device','auto','--workers','4','--envs-per-worker','4',
        '--seed',str(TRAINING_SEED),'--total-steps',str(TOTAL_STEPS),'--checkpoint-dir',LOCAL_CHECKPOINT_DIR,
        '--recovery-dir',RECOVERY,'--save-every-minutes','30'] + resume_args
print('Training', EXPERIMENT_ID, 'from', train_script, 'on branch', checked_out, 'mode:', resume_mode)
subprocess.run(args, cwd=project_root, check=True)


In [ ]:
# Validation only. Do not put sealed-final seeds in this notebook.
import hashlib, json, shutil, subprocess, sys
from pathlib import Path
pointer = json.loads((Path(RECOVERY) / 'latest.json').read_text())
checkpoint = Path(RECOVERY) / pointer['checkpoint']
assert checkpoint.is_file()
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == pointer['sha256']
export_dir = Path('/content/export') / EXPERIMENT_ID
export_dir.mkdir(parents=True, exist_ok=True)
candidate = export_dir / 'Robust_RL_v1.js'
validation_raw = export_dir / 'validation.jsonl'
validation_stats = export_dir / 'validation_stats.json'
runtime_json = export_dir / 'runtime.json'
subprocess.run([sys.executable,'bot-dev/rl/export_policy.py',str(checkpoint),str(candidate)], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable,'bot-dev/rl/export_policy_test.py',str(checkpoint),str(candidate)], cwd=PROJECT_ROOT, check=True)
subprocess.run(['node','bot-dev/rl/export_env_smoke.mjs',str(candidate)], cwd=PROJECT_ROOT, check=True)
subprocess.run(['node','bot-dev/rl/eval/paired_eval.mjs',f'--candidate={candidate}',f'--output={validation_raw}'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable,'bot-dev/rl/eval/stats.py',str(validation_raw),'--output',str(validation_stats)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
with runtime_json.open('w') as output:
    subprocess.run(['node','--expose-gc','bot-dev/rl/eval/runtime_bench.mjs',f'--candidate={candidate}'], cwd=PROJECT_ROOT, check=True, stdout=output)
artifact_dir = Path('/content/drive/MyDrive/pikachu_rl/evaluations') / f"{EXPERIMENT_ID}_step{pointer['globalStep']}"
artifact_dir.mkdir(parents=True, exist_ok=True)
for source in export_dir.iterdir():
    if source.is_file():
        shutil.copy2(source, artifact_dir / source.name)
stats = json.loads(validation_stats.read_text())
runtime = json.loads(runtime_json.read_text())
summary = {'checkpoint': pointer, 'candidate': stats['candidate'], 'v4': stats['v4'], 'nonBenchmarkPaired': stats['primary'], 'v4DirectPaired': stats['benchmarkPaired'], 'selfDestruction': stats['selfDestruction'], 'meanRallyFrames': stats['meanRallyFrames'], 'runtimeCandidate': runtime['candidate'], 'savedTo': str(artifact_dir)}
print(json.dumps(summary, indent=2))
print('Do not copy to src/code-here unless the pre-registered acceptance gate passes.')
